# GGUF vision models: zero-shot VQA evaluation on Kaggle, two GPUs

Import this notebook into Kaggle. Set **Accelerator → GPU T4 x2**, enable
**Internet**, and attach your dataset. Edit the configuration below, then
run all cells. The notebook bundles the repository evaluator; no repository
clone or upload is required.

Defaults use **PlantExpertVQA** and its existing Kaggle dataset path.
For **GQA**, set `DATASET = "gqa"` and change `QUESTIONS` and `IMAGES`.
Questions accept repository CSV (`image,question,answer`) or official GQA
question JSON. Answers are optional; unlabeled data cannot produce accuracy.

**To switch models, change only `MODEL_DIR`.** Use a Hugging Face GGUF
repository ID (or its URL) or a local Kaggle folder containing the GGUF
weights and matching vision projector. The runner discovers file names,
selects a quantization, and creates a separate results folder per model.
The default is the [official Ministral GGUF repository](https://huggingface.co/mistralai/Ministral-3-8B-Instruct-2512-GGUF/tree/main).
Automatic quantization prefers Q4_K_M, then Q5_K_M, then Q8_0. A single
available quantization is also accepted. For a specific HF quantization,
use `MODEL_DIR = "owner/model-GGUF:Q8_0"`. Local folders must contain one
model family and its projector (`mmproj` or `projector` in the file name).
VQA requires a vision model supported by the selected llama.cpp version.
[llama.cpp](https://github.com/ggml-org/llama.cpp/blob/v0.6.0/tools/server/README.md)
splits layers and KV cache across GPUs with `--split-mode layer --tensor-split 1,1`.
This is one model distributed over two GPUs; questions run sequentially.
A five-question smoke test checks the workflow before the full evaluation.
It does not establish benchmark performance.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

DATASET = "plantexpert"  # "plantexpert" or "gqa"
DATA_ROOT = Path("/kaggle/input/datasets/duyho0511chill/plantexpert-dataset/plantexpert_dataset")
QUESTIONS = DATA_ROOT / "test.csv"
IMAGES = DATA_ROOT / "images"
IMAGE_SPLIT = "test"  # Used for flat image names in a split subfolder.
MODEL_DIR = "mistralai/Ministral-3-8B-Instruct-2512-GGUF"  # Only model setting to change.
# Local example: MODEL_DIR = "/kaggle/input/my-vision-model/gguf"
HF_REVISION = "main"  # Prefer a fixed HF commit for repeated comparisons.
LLAMA_CPP_REF = "v0.6.0"
CONTEXT_SIZE = 8192
MAX_TOKENS = 256 if DATASET == "plantexpert" else 64
NO_BERTSCORE = False  # PlantExpert only: True skips the semantic metric.
RESUME = False  # Set True to resume the FULL run with identical settings.
FULL_LIMIT = None  # None = all questions; positive integer = first N.
WORK = Path("/kaggle/working")
EVAL_ROOT = WORK / "ot-vqa-eval"
RESULTS_ROOT = WORK / "results"  # Model-specific folder is computed automatically.
assert DATASET in {"gqa", "plantexpert"}
assert QUESTIONS.is_file(), f"Set QUESTIONS to your attached dataset: {QUESTIONS}"
assert IMAGES.is_dir(), f"Set IMAGES to your attached dataset: {IMAGES}"
print("Model:", MODEL_DIR, "\nQuestions:", QUESTIONS, "\nImages:", IMAGES)


## Install evaluation dependencies and check both GPUs

PlantExpert BERTScore runs on CPU to leave GPU memory for inference.
GQA scoring needs only Python's standard library. The first BERTScore run
downloads its scoring encoder; the GGUF download also needs Internet.


In [ ]:
packages = ["huggingface-hub>=0.34,<2"]
if DATASET == "plantexpert" and not NO_BERTSCORE:
    packages.append("bert-score==0.3.13")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
import torch
assert torch.cuda.device_count() >= 2, "Select GPU T4 x2 in Kaggle settings."
for index in range(2):
    device = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {device.name}, {device.total_memory / 2**30:.1f} GiB")
CUDA_ARCHS = ";".join(sorted({
    "".join(map(str, torch.cuda.get_device_capability(index))) for index in range(2)
}))


## Restore the bundled evaluation scripts

These files are a snapshot of this repository, written into Kaggle's
writable working directory. Regenerate the notebook after editing the
evaluator with `python3 -m scripts.build_kaggle_notebook`.


In [ ]:
BUNDLED_SOURCES = {'scripts/evaluate_gqa_lmstudio.py': '"""Zero-shot GQA or PlantExpertVQA evaluation through LM Studio.\n\nRun from the repository root:\n    python3 -m scripts.evaluate_gqa_lmstudio --limit 5 --output-dir results/qwen_smoke\n    python3 -m scripts.evaluate_gqa_lmstudio\n    python3 -m scripts.evaluate_gqa_lmstudio --dataset plantexpert\n\nGQA and PlantExpert lexical scoring use only the standard library.\nPlantExpert BERTScore additionally requires bert-score and its dependencies.\n\nEach request contains only an image and question, with no demonstrations,\nreference answers, scene graphs, or chat history. Images retain their original\nresolution; LM Studio handles the model\'s image preprocessing.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport csv\nimport hashlib\nimport json\nimport mimetypes\nimport os\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nfrom urllib.parse import urlsplit\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nfrom utils.metrics import PAPER_METRICS, build_bertscore_scorer, lexical_scores, score_pairs\n\n\nSYSTEM_PROMPT = (\n    "Answer the question using the image. Return only the short answer, "\n    "usually one word or a short phrase. For yes/no questions, return yes or no. "\n    "Do not include an explanation, a full sentence, or an \'Answer:\' prefix."\n)\nPLANTEXPERT_SYSTEM_PROMPT = (\n    "Answer the plant-science question using the image. Give a concise, complete "\n    "answer with the details requested by the question. For yes/no questions, "\n    "answer yes or no and explain only if requested. Do not include an \'Answer:\' "\n    "prefix, unrelated information, or discussion of your reasoning process."\n)\nMETADATA_FIELDS = ("question_type", "question_category", "crop", "disease")\n\n\n@dataclass(frozen=True)\nclass Example:\n    question_id: str\n    image: str\n    question: str\n    answer: str | None\n    metadata: dict[str, str] = field(default_factory=dict)\n\n\ndef load_examples(path: Path) -> list[Example]:\n    """Accept repository CSVs or official GQA questionId -> question JSON."""\n    examples = []\n    if path.suffix.lower() == ".csv":\n        with path.open(encoding="utf-8-sig", newline="") as handle:\n            reader = csv.DictReader(handle)\n            if not {"image", "question"}.issubset(reader.fieldnames or []):\n                raise ValueError("CSV must contain image and question columns; answer is optional.")\n            for index, row in enumerate(reader):\n                examples.append(Example(\n                    row.get("question_id") or row.get("anno_id") or str(index),\n                    row["image"], row["question"], row.get("answer"),\n                    {key: row[key] for key in METADATA_FIELDS if key in row},\n                ))\n    elif path.suffix.lower() == ".json":\n        data = json.loads(path.read_text(encoding="utf-8"))\n        if not isinstance(data, dict):\n            raise ValueError("GQA JSON must map question IDs to question objects.")\n        for question_id, row in data.items():\n            examples.append(Example(\n                str(question_id), f"{row[\'imageId\']}.jpg", row["question"],\n                row.get("answer") or None,\n            ))\n    else:\n        raise ValueError("Questions must be a .csv or .json file.")\n    if not examples:\n        raise ValueError("The question file is empty.")\n    ids = set()\n    for example in examples:\n        if example.question_id in ids:\n            raise ValueError(f"Duplicate question ID: {example.question_id}")\n        ids.add(example.question_id)\n        if not example.image or not example.question.strip():\n            raise ValueError(f"Missing image or question for {example.question_id}")\n    return examples\n\n\ndef image_path(root: Path, example: Example, split: str) -> Path:\n    """Resolve image.jpg, split/image.jpg, and repository split-folder layouts."""\n    direct = root / example.image\n    if direct.is_file():\n        return direct\n    nested = root / split / example.image\n    if nested.is_file():\n        return nested\n    raise FileNotFoundError(f"Image not found: {direct} (also checked {nested})")\n\n\ndef image_data_url(path: Path) -> str:\n    mime = mimetypes.guess_type(str(path))[0]\n    if mime not in {"image/jpeg", "image/png", "image/webp", "image/gif"}:\n        raise ValueError(f"Unsupported image format: {path}")\n    return f"data:{mime};base64,{base64.b64encode(path.read_bytes()).decode(\'ascii\')}"\n\n\nclass LMStudioClient:\n    def __init__(self, base_url: str, timeout: float, retries: int,\n                 server_name: str = "LM Studio"):\n        self.base_url = base_url.rstrip("/")\n        if not self.base_url.endswith("/v1"):\n            self.base_url += "/v1"\n        self.timeout = timeout\n        self.retries = retries\n        self.server_name = server_name\n\n    def request(self, endpoint: str, payload: dict | None = None) -> dict:\n        headers = {"Content-Type": "application/json"}\n        # Set this only when authentication is enabled in LM Studio.\n        if os.environ.get("LM_STUDIO_API_KEY"):\n            headers["Authorization"] = "Bearer " + os.environ["LM_STUDIO_API_KEY"]\n        request = urllib.request.Request(\n            self.base_url + endpoint,\n            data=None if payload is None else json.dumps(payload).encode("utf-8"),\n            headers=headers,\n        )\n        for attempt in range(self.retries + 1):\n            try:\n                with urllib.request.urlopen(request, timeout=self.timeout) as response:\n                    return json.load(response)\n            except urllib.error.HTTPError as error:\n                detail = error.read().decode("utf-8", errors="replace")[:1000]\n                if error.code not in {429, 500, 502, 503, 504} or attempt == self.retries:\n                    raise RuntimeError(f"{self.server_name} HTTP {error.code}: {detail}") from error\n            except (urllib.error.URLError, TimeoutError, ConnectionError) as error:\n                if attempt == self.retries:\n                    raise RuntimeError(\n                        f"Cannot reach {self.base_url}: {error}. "\n                        f"Check that the {self.server_name} server is running and the vision model is loaded."\n                    ) from error\n            time.sleep(min(2 ** attempt, 8))\n        raise RuntimeError("Request exhausted its retry attempts.")\n\n    def models(self) -> list[str]:\n        return [item["id"] for item in self.request("/models")["data"]]\n\n    def answer(self, model: str, example: Example, path: Path,\n               max_tokens: int, seed: int, system_prompt: str = SYSTEM_PROMPT) -> dict:\n        start = time.perf_counter()\n        response = self.request("/chat/completions", {\n            "model": model,\n            "messages": [\n                {"role": "system", "content": system_prompt},\n                {"role": "user", "content": [\n                    {"type": "image_url", "image_url": {"url": image_data_url(path)}},\n                    {"type": "text", "text": example.question},\n                ]},\n            ],\n            "temperature": 0,\n            "seed": seed,\n            "max_tokens": max_tokens,\n            "stream": False,\n        })\n        choice = response["choices"][0]\n        raw = choice["message"].get("content")\n        if not isinstance(raw, str) or not raw.strip():\n            raise RuntimeError("The model returned no answer text. Check max-tokens and model settings.")\n        return {\n            "raw_response": raw,\n            "prediction": raw.strip(),\n            "finish_reason": choice.get("finish_reason"),\n            "latency_seconds": time.perf_counter() - start,\n            "usage": response.get("usage", {}),\n        }\n\n\ndef normalize(text: str) -> str:\n    return " ".join(text.casefold().split())\n\n\ndef make_record(example: Example, result: dict | None = None, error: str = "",\n                dataset: str = "gqa") -> dict:\n    record = {\n        "question_id": example.question_id, "image": example.image,\n        "question": example.question, "reference": example.answer,\n        "prediction": "", "raw_response": "", "error": error,\n    }\n    record.update(example.metadata)\n    record.update(result or {})\n    labeled = example.answer is not None\n    if dataset == "plantexpert":\n        if not labeled:\n            scores = {metric: None for metric in PAPER_METRICS[:-1]}\n        elif error:\n            scores = {metric: 0.0 for metric in PAPER_METRICS[:-1]}\n        else:\n            scores = lexical_scores(example.answer, record["prediction"])\n        record.update(scores)\n        return record\n    record["correct"] = (\n        bool(not error and record["prediction"].strip() == example.answer.strip()) if labeled else None\n    )\n    record["normalized_correct"] = (\n        bool(not error and normalize(record["prediction"]) == normalize(example.answer))\n        if labeled else None\n    )\n    return record\n\n\ndef save_outputs(output: Path, records: dict[str, dict], config: dict,\n                 selected_count: int, bert_scorer=None) -> dict:\n    rows = list(records.values())\n    labeled = [row for row in rows if row["reference"] is not None]\n    successful = [row for row in rows if not row["error"]]\n    report = {\n        "config": config,\n        "examples_selected": selected_count,\n        "examples_attempted": len(rows),\n        "examples_succeeded": len(successful),\n        "errors": len(rows) - len(successful),\n        "labeled_examples_attempted": len(labeled),\n        "complete": len(successful) == selected_count,\n        "truncated_responses": sum(row.get("finish_reason") == "length" for row in successful),\n        "mean_latency_seconds": (\n            sum(row["latency_seconds"] for row in successful) / len(successful)\n            if successful else None\n        ),\n    }\n    plantexpert = config.get("dataset") == "plantexpert"\n    if plantexpert:\n        report["generated_metrics"] = {\n            metric: sum(row[metric] for row in labeled) / len(labeled) if labeled else None\n            for metric in PAPER_METRICS[:-1]\n        }\n        report["generated_metrics"]["bertscore_f1"] = None\n        report["bertscore"] = config["bertscore"].copy()\n        report["scoring"] = {\n            "metrics": "Repository PlantExpert metrics; scores are fractions, rescaled BERTScore can be negative.",\n            "errors": "Failed labeled requests receive zero for every metric; unattempted questions are excluded.",\n        }\n    else:\n        report.update({\n            "accuracy": sum(row["correct"] for row in labeled) / len(labeled) if labeled else None,\n            "normalized_accuracy": (\n                sum(row["normalized_correct"] for row in labeled) / len(labeled) if labeled else None\n            ),\n            "scoring": {\n                "accuracy": "Exact match after trimming response whitespace; one point per correct answer.",\n                "normalized_accuracy": "Exact match ignoring case and repeated whitespace.",\n                "errors": "Failed labeled requests count as incorrect; unattempted questions are excluded.",\n            },\n        })\n    # Save a report before semantic scoring, so scoring failures preserve inference results.\n    (output / "report.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n    fields = ["question_id", "image", "question", "reference", "prediction"]\n    fields += list(PAPER_METRICS) if plantexpert else ["correct", "normalized_correct"]\n    fields += [key for key in METADATA_FIELDS if any(key in row for row in rows)]\n    fields += ["latency_seconds", "finish_reason", "error"]\n\n    def write_csv():\n        with (output / "predictions.csv").open("w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n            writer.writeheader()\n            writer.writerows(rows)\n\n    write_csv()\n    if plantexpert:\n        for row in rows:\n            failed_label = row["error"] and row["reference"] is not None\n            row["bertscore_f1"] = 0.0 if bert_scorer is not None and failed_label else None\n        if bert_scorer is not None:\n            scored = [row for row in labeled if not row["error"]]\n            if scored:\n                scores = score_pairs([row["reference"] for row in scored],\n                                     [row["prediction"] for row in scored], bert_scorer)\n                for row, metrics in zip(scored, scores):\n                    row["bertscore_f1"] = metrics["bertscore_f1"]\n            report["generated_metrics"]["bertscore_f1"] = (\n                sum(row["bertscore_f1"] for row in labeled) / len(labeled) if labeled else None\n            )\n            report["bertscore"]["model_hash"] = bert_scorer.hash\n        (output / "report.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        write_csv()\n        return report\n    # Official GQA prediction structure. Repository anno_id values are image IDs,\n    # so use official question JSON if exporting predictions for the GQA evaluator.\n    (output / "gqa_predictions.json").write_text(json.dumps([\n        {"questionId": row["question_id"], "prediction": row["prediction"]}\n        for row in successful\n    ], indent=2) + "\\n", encoding="utf-8")\n    return report\n\n\ndef evaluate(args: argparse.Namespace, client: LMStudioClient) -> dict:\n    examples = load_examples(args.questions)\n    if args.limit is not None:\n        examples = examples[:args.limit]\n    # Validate the selected files before starting a potentially long inference run.\n    paths = {example.question_id: image_path(args.images, example, args.image_split or args.questions.stem)\n             for example in examples}\n    plantexpert = args.dataset == "plantexpert"\n    system_prompt = PLANTEXPERT_SYSTEM_PROMPT if plantexpert else SYSTEM_PROMPT\n    config = {\n        "questions": str(args.questions.resolve()),\n        "questions_sha256": hashlib.sha256(args.questions.read_bytes()).hexdigest(),\n        "images": str(args.images.resolve()),\n        "image_split": args.image_split or args.questions.stem,\n        "model": args.model, "base_url": client.base_url,\n        "limit": args.limit, "seed": args.seed,\n        "temperature": 0, "max_tokens": args.max_tokens,\n        "system_prompt": system_prompt, "protocol": "zero-shot",\n    }\n    if getattr(args, "backend_config", None) is not None:\n        config["backend"] = args.backend_config\n    if plantexpert:\n        config.update({"dataset": "plantexpert", "bertscore": {\n            "enabled": not args.no_bertscore, "model": args.bertscore_model,\n            "device": args.bertscore_device, "batch_size": args.bertscore_batch_size,\n            "rescale_with_baseline": args.bertscore_rescale,\n        }})\n    output = args.output_dir\n    output.mkdir(parents=True, exist_ok=True)\n    manifest = output / "run_config.json"\n    journal = output / "predictions.jsonl"\n    records = {}\n    if args.resume:\n        saved_config = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else None\n        # Managed llama.cpp runs can use a fresh local port without changing inference.\n        # Keep all model, dataset, prompt, and backend checks intact.\n        if saved_config is not None and getattr(args, "allow_server_port_change", False):\n            if (saved_config.get("backend", {}).get("engine") == "llama.cpp"\n                    and config.get("backend", {}).get("engine") == "llama.cpp"):\n                before = urlsplit(saved_config.get("base_url", ""))\n                after = urlsplit(config["base_url"])\n                if (before.scheme == after.scheme == "http"\n                        and before.hostname == after.hostname == "127.0.0.1"\n                        and before.path == after.path == "/v1"\n                        and not any((before.username, after.username, before.query, after.query,\n                                     before.fragment, after.fragment))):\n                    saved_config["base_url"] = config["base_url"]\n        if saved_config != config:\n            raise ValueError("Resume configuration differs or is missing. Use the original flags or a new output-dir.")\n        if journal.is_file():\n            for line in journal.read_text(encoding="utf-8").splitlines():\n                row = json.loads(line)\n                records[row["question_id"]] = row\n    else:\n        if any(output.iterdir()):\n            raise ValueError(f"Output directory is not empty: {output}. Use --resume or a new output-dir.")\n    bert_scorer = None\n    if plantexpert and not args.no_bertscore and any(example.answer is not None for example in examples):\n        print("Loading BERTScore before inference...", flush=True)\n        bert_scorer = build_bertscore_scorer(\n            model_type=args.bertscore_model, device=args.bertscore_device,\n            batch_size=args.bertscore_batch_size, rescale_with_baseline=args.bertscore_rescale,\n        )\n    if not args.resume:\n        manifest.write_text(json.dumps(config, indent=2) + "\\n", encoding="utf-8")\n    print(f"Model: {args.model}; selected questions: {len(examples)}; zero-shot", flush=True)\n    try:\n        with journal.open("a", encoding="utf-8") as handle:\n            for index, example in enumerate(examples, 1):\n                previous = records.get(example.question_id)\n                if previous is not None and not previous["error"]:\n                    continue\n                failure = None\n                try:\n                    prompt_args = {"system_prompt": system_prompt} if plantexpert else {}\n                    result = client.answer(args.model, example, paths[example.question_id],\n                                           args.max_tokens, args.seed, **prompt_args)\n                    record = make_record(example, result, dataset=args.dataset)\n                except (RuntimeError, OSError, ValueError, KeyError, IndexError, TypeError) as error:\n                    failure = error\n                    record = make_record(example, error=str(error), dataset=args.dataset)\n                handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                handle.flush()\n                records[example.question_id] = record\n                print(f"[{index}/{len(examples)}] {example.question_id}: "\n                      f"{record[\'prediction\']!r} | reference={example.answer!r}"\n                      + (f" | ERROR: {failure}" if failure else ""), flush=True)\n                if failure and not args.continue_on_error:\n                    raise RuntimeError("Stopped after an error; predictions are saved. Fix it and use --resume.") from failure\n    finally:\n        report = save_outputs(output, records, config, len(examples), bert_scorer)\n    return report\n\n\ndef parse_args(argv: list[str] | None = None) -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--base-url", default="http://127.0.0.1:1234/v1")\n    parser.add_argument("--model", default="qwen3-vl-2b-instruct", help="Exact identifier from --list-models")\n    parser.add_argument("--list-models", action="store_true", help="Print model identifiers and exit")\n    parser.add_argument("--dataset", choices=("gqa", "plantexpert"), default="gqa",\n                        help="Choose dataset defaults, answer prompt, and scoring (default: gqa)")\n    parser.add_argument("--questions", type=Path, help="Default: data/<dataset>_dataset/test.csv")\n    parser.add_argument("--images", type=Path, help="Default: data/<dataset>_dataset/images")\n    parser.add_argument("--image-split", help="Optional split subfolder for flat image names")\n    parser.add_argument("--output-dir", type=Path, help="Default: results/qwen3_vl_2b_<dataset>_zero_shot")\n    parser.add_argument("--limit", type=int, help="Evaluate only the first N questions (default: all)")\n    parser.add_argument("--max-tokens", type=int, help="Default: 64 for GQA, 256 for PlantExpert")\n    parser.add_argument("--no-bertscore", action="store_true",\n                        help="PlantExpert: compute five lexical metrics only, without extra dependencies")\n    parser.add_argument("--bertscore-model", default="bert-base-uncased")\n    parser.add_argument("--bertscore-device", default="cpu")\n    parser.add_argument("--bertscore-batch-size", type=int, default=16)\n    parser.add_argument("--no-bertscore-rescale", dest="bertscore_rescale", action="store_false",\n                        help="Disable BERTScore English baseline rescaling")\n    parser.add_argument("--seed", type=int, default=42, help="LM Studio sampling seed")\n    parser.add_argument("--timeout", type=float, default=180, help="Request timeout in seconds")\n    parser.add_argument("--retries", type=int, default=2, help="Additional attempts for transient API failures")\n    parser.add_argument("--resume", action="store_true", help="Reuse saved successes and retry failed questions")\n    parser.add_argument("--continue-on-error", action="store_true", help="Record errors and continue; errors count as wrong")\n    args = parser.parse_args(argv)\n    args.questions = args.questions or Path(f"data/{args.dataset}_dataset/test.csv")\n    args.images = args.images or Path(f"data/{args.dataset}_dataset/images")\n    args.output_dir = args.output_dir or Path(f"results/qwen3_vl_2b_{args.dataset}_zero_shot")\n    if args.max_tokens is None:\n        args.max_tokens = 256 if args.dataset == "plantexpert" else 64\n    if args.limit is not None and args.limit <= 0:\n        parser.error("--limit must be positive")\n    if args.max_tokens <= 0 or args.timeout <= 0 or args.retries < 0:\n        parser.error("max-tokens and timeout must be positive; retries must be nonnegative")\n    if args.bertscore_batch_size <= 0:\n        parser.error("bertscore-batch-size must be positive")\n    return args\n\n\ndef main(argv: list[str] | None = None) -> int:\n    args = parse_args(argv)\n    client = LMStudioClient(args.base_url, args.timeout, args.retries)\n    try:\n        models = client.models()\n        if args.list_models:\n            print("\\n".join(models))\n            return 0\n        if args.model not in models:\n            raise ValueError(f"Model {args.model!r} was not found. Available IDs: {\', \'.join(models)}")\n        report = evaluate(args, client)\n    except KeyboardInterrupt:\n        print("\\nInterrupted. Saved completed predictions; rerun with --resume.", file=sys.stderr)\n        return 130\n    except (RuntimeError, OSError, ValueError, KeyError, TypeError) as error:\n        print(f"Error: {error}", file=sys.stderr)\n        return 1\n    if args.dataset == "plantexpert":\n        print("PlantExpert metrics: " + ", ".join(\n            f"{key}={value:.4f}" if value is not None else f"{key}=not computed"\n            for key, value in report["generated_metrics"].items()\n        ))\n    elif report["accuracy"] is not None:\n        print(f"Accuracy: {report[\'accuracy\']:.2%}; normalized: {report[\'normalized_accuracy\']:.2%}")\n    else:\n        print("No reference answers supplied; predictions saved without accuracy scoring.")\n    print(f"Results: {args.output_dir / \'report.json\'}")\n    return 0 if report["complete"] else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'scripts/evaluate_kaggle_ministral.py': '"""Run the existing zero-shot evaluator with a local two-GPU llama.cpp server.\n\nBuild a CUDA-enabled llama-server first (see the accompanying Kaggle notebook).\nAll evaluation flags from scripts.evaluate_gqa_lmstudio are accepted.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport errno\nimport hashlib\nimport json\nimport os\nimport re\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nfrom contextlib import contextmanager\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nfrom scripts.evaluate_gqa_lmstudio import (\n    LMStudioClient, evaluate, image_path, load_examples, parse_args as evaluation_args,\n)\n\n\nDEFAULT_MODEL_DIR = "mistralai/Ministral-3-8B-Instruct-2512-GGUF"\n_QUANT = re.compile(r"(?:^|[-_.])(I?Q\\d[A-Z0-9_]*|BF16|FP16|F16|FP32|F32)(?=[-.]|$)", re.I)\n_SHARD = re.compile(r"-(\\d{5})-of-(\\d{5})\\.gguf$", re.I)\n\n\ndef model_identifier(model_dir: str, quantization: str | None = None) -> str:\n    """Create a stable alias and output folder from the single model setting."""\n    value = str(model_dir).rstrip("/")\n    name = re.sub(r"[^a-z0-9_-]+", "-", value.rsplit("/", 1)[-1].lower()).strip("-")\n    digest = hashlib.sha256((value + "|" + (quantization or "auto")).encode()).hexdigest()[:10]\n    return f"{name or \'gguf-model\'}-{digest}"\n\n\ndef quantization_of(name: str) -> str | None:\n    match = _QUANT.search(_SHARD.sub(".gguf", Path(name).name))\n    return match.group(1).upper() if match else None\n\n\ndef model_family(name: str) -> str:\n    stem = Path(_SHARD.sub(".gguf", name)).stem\n    stem = _QUANT.sub("", stem)\n    stem = re.sub(r"mmproj|projector", "", stem, flags=re.I)\n    return re.sub(r"[^a-z0-9]", "", stem.lower())\n\n\ndef select_gguf_files(names: list[str], quantization: str | None = None) -> tuple[str, str, list[str]]:\n    """Select one model family and its projector, including all weight shards."""\n    files = sorted(name for name in names if name.lower().endswith(".gguf"))\n    projectors = [name for name in files if re.search(r"mmproj|projector", Path(name).name, re.I)]\n    weights = [name for name in files if name not in projectors]\n    candidates = [name for name in weights if not _SHARD.search(name) or int(_SHARD.search(name).group(1)) == 1]\n    if not candidates:\n        raise ValueError("MODEL_DIR must contain GGUF model weights (or the first shard).")\n    if not projectors:\n        raise ValueError("VQA requires a vision GGUF projector named mmproj or projector in MODEL_DIR.")\n    if len({model_family(name) for name in candidates}) > 1:\n        raise ValueError("MODEL_DIR contains multiple model families. Use a folder or HF repository for one model.")\n    requested = quantization.upper() if quantization else None\n    if requested:\n        candidates = [name for name in candidates if quantization_of(name) == requested]\n    else:\n        for preferred in ("Q4_K_M", "Q5_K_M", "Q8_0", "Q4_0", "F16", "FP16", "BF16"):\n            matches = [name for name in candidates if quantization_of(name) == preferred]\n            if matches:\n                candidates = matches\n                break\n    if len(candidates) != 1:\n        raise ValueError(f"Cannot select unique GGUF weights ({requested or \'auto\'}): {candidates}. "\n                         "Use a directory with one quantization, or append :Q4_K_M to the HF repo ID.")\n    model = candidates[0]\n    matching = [name for name in projectors if model_family(name) == model_family(model)]\n    projectors = matching or [name for name in projectors if not model_family(name)]\n    if not projectors:\n        raise ValueError("No projector matches the selected model. Keep its matching mmproj in MODEL_DIR.")\n    if len(projectors) > 1:\n        for preferred in ("F16", "FP16", "BF16", "Q8_0"):\n            matches = [name for name in projectors if quantization_of(name) == preferred]\n            if matches:\n                projectors = matches\n                break\n    if len(projectors) != 1:\n        raise ValueError(f"Multiple matching vision projectors: {projectors}. Use a folder with one projector.")\n    shards = [model]\n    match = _SHARD.search(model)\n    if match:\n        shards = [_SHARD.sub(f"-{index:05d}-of-{int(match.group(2)):05d}.gguf", model)\n                  for index in range(1, int(match.group(2)) + 1)]\n        missing = set(shards) - set(weights)\n        if missing:\n            raise ValueError(f"Missing GGUF weight shards: {sorted(missing)}")\n    return model, projectors[0], shards\n\n\ndef gpu_environment() -> tuple[dict, list[str]]:\n    """Respect CUDA visibility and restrict this run to the first two devices."""\n    try:\n        result = subprocess.run(\n            ["nvidia-smi", "--query-gpu=uuid,name,memory.total", "--format=csv,noheader"],\n            check=True, capture_output=True, text=True,\n        )\n    except (OSError, subprocess.CalledProcessError) as error:\n        raise RuntimeError("Enable a two-GPU Kaggle accelerator before running.") from error\n    devices = [line.strip() for line in result.stdout.splitlines() if line.strip()]\n    visible = os.environ.get("CUDA_VISIBLE_DEVICES")\n    if visible is None:\n        selected = ["0", "1"] if len(devices) >= 2 else []\n    else:\n        selected = [item.strip() for item in visible.split(",") if item.strip()]\n        if any(item == "-1" for item in selected):\n            selected = []\n        selected = selected[:2]\n    if len(devices) < 2 or len(selected) < 2 or len(set(selected)) != 2:\n        raise RuntimeError("Two visible NVIDIA GPUs are required; choose GPU T4 x2 in Kaggle.")\n    identifiers = {row.split(",", 1)[0].strip() for row in devices}\n    for identifier in selected:\n        valid_index = identifier.isdecimal() and int(identifier) < len(devices)\n        if not valid_index and identifier not in identifiers:\n            raise RuntimeError(f"CUDA_VISIBLE_DEVICES contains an unavailable GPU: {identifier}")\n    environment = os.environ.copy()\n    environment["CUDA_VISIBLE_DEVICES"] = ",".join(selected)\n    return environment, devices\n\n\ndef resolve_model(model_dir: str, cache_dir: Path, quantization: str | None = None,\n                  revision: str = "main") -> tuple[Path, Path, dict]:\n    """Accept a local folder or HF repo; discover weights and vision projector."""\n    local = Path(model_dir).expanduser()\n    if local.exists():\n        if not local.is_dir():\n            raise ValueError("MODEL_DIR must be a folder containing GGUF weights and a matching projector.")\n        names = [path.relative_to(local).as_posix() for path in local.rglob("*") if path.is_file()]\n        model, projector, shards = select_gguf_files(names, quantization)\n        checksums = {}\n        for name in [*shards, projector]:\n            digest = hashlib.sha256()\n            with (local / name).open("rb") as handle:\n                for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n                    digest.update(chunk)\n            checksums[name] = digest.hexdigest()\n        return local / model, local / projector, {\n            "directory": str(local.resolve()), "model_file": model, "projector_file": projector,\n            "quantization": quantization_of(model), "files_sha256": checksums,\n        }\n    if local.is_absolute() or str(model_dir).startswith((".", "~")):\n        raise FileNotFoundError(f"Local MODEL_DIR does not exist: {local}")\n    repo = str(model_dir).rstrip("/")\n    if repo.startswith("https://huggingface.co/"):\n        repo = urlparse(repo).path.strip("/")\n    if ":" in repo:\n        repo, selected_quant = repo.rsplit(":", 1)\n        if quantization and quantization.upper() != selected_quant.upper():\n            raise ValueError("Conflicting quantizations in MODEL_DIR and --quantization")\n        quantization = selected_quant\n    if not re.fullmatch(r"[\\w.-]+/[\\w.-]+", repo):\n        raise ValueError("MODEL_DIR must be a local GGUF folder or Hugging Face owner/model repository.")\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n    except ImportError as error:\n        raise RuntimeError("Install huggingface-hub before downloading GGUF files.") from error\n    api = HfApi()\n    resolved = api.model_info(repo, revision=revision).sha\n    model, projector, shards = select_gguf_files(api.list_repo_files(repo, revision=resolved), quantization)\n    destination = cache_dir / model_identifier(repo) / resolved\n    paths = {name: Path(hf_hub_download(\n        repo_id=repo, filename=name, revision=resolved, local_dir=str(destination),\n    )) for name in [*shards, projector]}\n    return paths[model], paths[projector], {\n        "repo": repo, "revision": resolved, "quantization": quantization_of(model),\n        "model_file": model, "projector_file": projector, "weight_files": shards,\n    }\n\n\ndef server_command(binary: Path, model: Path, projector: Path, alias: str,\n                   port: int, context_size: int) -> list[str]:\n    for artifact in (binary, model, projector):\n        if not artifact.is_file():\n            raise FileNotFoundError(f"Required file not found: {artifact}")\n    return [\n        str(binary.resolve()), "--model", str(model.resolve()),\n        "--mmproj", str(projector.resolve()), "--alias", alias,\n        "--host", "127.0.0.1", "--port", str(port),\n        "--n-gpu-layers", "99", "--split-mode", "layer", "--tensor-split", "1,1",\n        "--main-gpu", "0", "--ctx-size", str(context_size), "--parallel", "1",\n        "--batch-size", "512", "--ubatch-size", "128", "--flash-attn", "off", "--jinja",\n    ]\n\n\ndef wait_for_server(process, port: int, timeout: float):\n    deadline = time.monotonic() + timeout\n    while time.monotonic() < deadline:\n        if process.poll() is not None:\n            raise RuntimeError(f"llama-server exited with code {process.returncode}; inspect its log.")\n        try:\n            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2) as response:\n                if json.load(response).get("status") == "ok":\n                    return\n        except (urllib.error.URLError, TimeoutError, ConnectionError):\n            pass\n        time.sleep(1)\n    raise RuntimeError(f"llama-server did not become ready within {timeout:g}s; inspect its log.")\n\n\ndef available_server_port(port: int = 0) -> int:\n    """Reuse closing connections; port 0 asks the OS to select a free port."""\n    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:\n        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n        try:\n            probe.bind(("127.0.0.1", port))\n        except OSError as error:\n            if error.errno == errno.EADDRINUSE:\n                raise RuntimeError(\n                    f"Port {port} is occupied. Use --server-port 0 to select a free port."\n                ) from error\n            raise\n        return probe.getsockname()[1]\n\n\n@contextmanager\ndef running_server(command: list[str], environment: dict, log_path: Path,\n                   port: int, timeout: float):\n    # Refuse an occupied port so health checks cannot attach to a different model.\n    available_server_port(port)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    with log_path.open("w", encoding="utf-8") as log:\n        process = subprocess.Popen(command, env=environment, stdout=log, stderr=subprocess.STDOUT)\n        try:\n            wait_for_server(process, port, timeout)\n            yield process\n        finally:\n            if process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=15)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait()\n\n\ndef parse_args(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__, add_help=False, allow_abbrev=False)\n    parser.add_argument("--llama-server", type=Path,\n                        default=Path("/kaggle/working/llama.cpp/build/bin/llama-server"))\n    parser.add_argument("--model-dir", default=DEFAULT_MODEL_DIR,\n                        help="Local GGUF folder or HF repo ID; optional repo:Q8_0 quantization suffix")\n    parser.add_argument("--cache-dir", type=Path, default=Path("/kaggle/working/gguf-models"))\n    parser.add_argument("--quantization", help="Optional quantization override; auto prefers Q4_K_M")\n    parser.add_argument("--hf-revision", default="main", help="HF commit or revision to download")\n    parser.add_argument("--context-size", type=int, default=8192)\n    parser.add_argument("--server-port", type=int, default=0,\n                        help="Local port; 0 automatically chooses a free port (default)")\n    parser.add_argument("--startup-timeout", type=float, default=300)\n    parser.add_argument("--server-log", type=Path, default=Path("/kaggle/working/llama-server.log"))\n    options, remaining = parser.parse_known_args(argv)\n    if "--help" in remaining or "-h" in remaining:\n        print(parser.format_help())\n        evaluation_args(["--help"])\n    if options.context_size <= 0 or options.startup_timeout <= 0 or not 0 <= options.server_port <= 65535:\n        parser.error("context-size and startup-timeout must be positive; server-port must be 0..65535")\n    alias = model_identifier(options.model_dir, options.quantization)\n    url = f"http://127.0.0.1:{options.server_port}/v1"\n    args = evaluation_args(["--model", alias, "--base-url", url, *remaining])\n    if args.base_url.rstrip("/") != url:\n        parser.error("The local server URL is managed by --server-port; omit --base-url")\n    if "--output-dir" not in remaining and not any(flag.startswith("--output-dir=") for flag in remaining):\n        args.output_dir = Path(f"/kaggle/working/results/{alias}_{args.dataset}")\n    return options, args\n\n\ndef main(argv=None) -> int:\n    options, args = parse_args(argv)\n    try:\n        environment, devices = gpu_environment()\n        if not options.llama_server.is_file():\n            raise FileNotFoundError(f"Build CUDA llama-server first: {options.llama_server}")\n        # Fail on bad dataset paths before a multi-GB download.\n        examples = load_examples(args.questions)\n        for example in examples[:args.limit]:\n            image_path(args.images, example, args.image_split or args.questions.stem)\n        print("NVIDIA GPUs: " + "; ".join(devices), flush=True)\n        model, projector, source = resolve_model(options.model_dir, options.cache_dir,\n                                               options.quantization, options.hf_revision)\n        print(f"Selected weights: {model.name}; vision projector: {projector.name}", flush=True)\n        options.server_port = available_server_port(options.server_port)\n        args.base_url = f"http://127.0.0.1:{options.server_port}/v1"\n        command = server_command(options.llama_server, model, projector, args.model,\n                                 options.server_port, options.context_size)\n        version = subprocess.run([str(options.llama_server.resolve()), "--version"],\n                                 check=True, capture_output=True, text=True)\n        args.backend_config = {\n            "engine": "llama.cpp", "version": (version.stdout + version.stderr).strip(),\n            "model_source": source, "split_mode": "layer", "tensor_split": "1,1",\n            "context_size": options.context_size, "parallel": 1, "flash_attention": "off",\n            "cuda_visible_devices": environment["CUDA_VISIBLE_DEVICES"],\n        }\n        args.allow_server_port_change = True\n        print(f"Starting llama-server on two GPUs at {args.base_url}; log: {options.server_log}", flush=True)\n        with running_server(command, environment, options.server_log,\n                            options.server_port, options.startup_timeout):\n            client = LMStudioClient(args.base_url, args.timeout, args.retries, server_name="llama.cpp")\n            if args.model not in client.models():\n                raise RuntimeError("llama-server did not expose the configured model alias.")\n            if args.list_models:\n                print("\\n".join(client.models()))\n                return 0\n            report = evaluate(args, client)\n        print(json.dumps(report, indent=2), flush=True)\n        print(f"Results: {args.output_dir}", flush=True)\n        return 0 if report["complete"] else 1\n    except KeyboardInterrupt:\n        print("Interrupted. Rerun with identical settings and --resume.", file=sys.stderr)\n        return 130\n    except (RuntimeError, OSError, ValueError, KeyError, TypeError, subprocess.SubprocessError) as error:\n        print(f"Error: {error}\\nServer log: {options.server_log}", file=sys.stderr)\n        return 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'utils/metrics.py': '"""Dataset-specific generated-answer metrics for GQA and PlantExpertVQA.\n\nScores are fractions (0 to 1 for lexical metrics), not percentages. BERTScore\ncan be negative when baseline rescaling is enabled. The paper gives formulas\nbut not its tokenizer or BERTScore encoder; those choices are explicit here.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nimport string\nfrom collections import Counter\nfrom statistics import fmean\n\n\nPAPER_METRICS = ("em", "token_f1", "bleu_1", "bleu_2", "rouge_l", "bertscore_f1")\nGQA_METRICS = ("accuracy",)\n_ARTICLE = re.compile(r"\\b(a|an|the)\\b", flags=re.IGNORECASE)\n_WORD_PUNCT = re.compile(r"\\w+|[^\\w\\s]", flags=re.UNICODE)\n\n\ndef resolve_dataset(dataset="auto", csv_path=None):\n    """Use an explicit dataset or detect GQA in a CSV path; otherwise paper metrics."""\n    dataset = dataset.lower()\n    if dataset not in {"auto", "gqa", "plantexpert"}:\n        raise ValueError(f"Unknown dataset: {dataset}")\n    if dataset != "auto":\n        return dataset\n    return ("gqa" if csv_path is not None and\n            re.search(r"(?:^|[^a-z0-9])gqa(?:$|[^a-z0-9])", str(csv_path).lower())\n            else "plantexpert")\n\n\ndef metrics_for_dataset(dataset):\n    return GQA_METRICS if resolve_dataset(dataset) == "gqa" else PAPER_METRICS\n\n\ndef normalize_text(text):\n    """Light normalization retained for dataset diagnostics."""\n    return " ".join(str(text).casefold().strip().split())\n\n\ndef gqa_accuracy(reference, hypothesis):\n    """Binary GQA exact match against one reference, with surrounding whitespace trimmed.\n\n    A matching generated answer receives 1 and a wrong answer receives 0.\n    Case, punctuation, and internal whitespace remain significant; answers are\n    not extracted from prose, mapped to synonyms, or scored by annotator votes.\n    """\n    if not isinstance(reference, str) or not isinstance(hypothesis, str):\n        raise TypeError("GQA accuracy requires one reference string and one prediction string")\n    return float(reference.strip() == hypothesis.strip())\n\n\ndef gqa_score_pairs(references, hypotheses):\n    """Per-example binary accuracy for aligned GQA references and predictions."""\n    if len(references) != len(hypotheses):\n        raise ValueError("References and hypotheses must have equal lengths")\n    return [{"accuracy": gqa_accuracy(ref, hyp)}\n            for ref, hyp in zip(references, hypotheses)]\n\n\ndef _as_text(value):\n    return value if isinstance(value, str) else " ".join(value)\n\n\ndef _squad_tokens(text):\n    text = _as_text(text).lower()\n    text = "".join(character for character in text if character not in string.punctuation)\n    text = _ARTICLE.sub(" ", text)\n    return text.split()\n\n\ndef _word_tokens(text):\n    return _WORD_PUNCT.findall(_as_text(text).casefold())\n\n\ndef _token_f1(reference, hypothesis):\n    ref, hyp = _squad_tokens(reference), _squad_tokens(hypothesis)\n    if not ref or not hyp:\n        return float(ref == hyp)\n    common = sum((Counter(ref) & Counter(hyp)).values())\n    return 2 * common / (len(ref) + len(hyp))\n\n\ndef _bleu(reference, hypothesis, order):\n    ref, hyp = _word_tokens(reference), _word_tokens(hypothesis)\n    if not ref or not hyp:\n        return float(ref == hyp)\n    precisions = []\n    for n in range(1, order + 1):\n        ref_ngrams = Counter(tuple(ref[i:i + n]) for i in range(len(ref) - n + 1))\n        hyp_ngrams = Counter(tuple(hyp[i:i + n]) for i in range(len(hyp) - n + 1))\n        if not hyp_ngrams:\n            return 0.0\n        matches = sum((ref_ngrams & hyp_ngrams).values())\n        if matches == 0:\n            return 0.0\n        precisions.append(matches / sum(hyp_ngrams.values()))\n    brevity_penalty = min(1.0, math.exp(1 - len(ref) / len(hyp)))\n    return brevity_penalty * math.exp(sum(math.log(p) for p in precisions) / order)\n\n\ndef _rouge_l(reference, hypothesis):\n    ref, hyp = _word_tokens(reference), _word_tokens(hypothesis)\n    if not ref or not hyp:\n        return float(ref == hyp)\n    previous = [0] * (len(hyp) + 1)\n    for ref_token in ref:\n        current = [0]\n        for column, hyp_token in enumerate(hyp, 1):\n            current.append(previous[column - 1] + 1 if ref_token == hyp_token\n                           else max(previous[column], current[-1]))\n        previous = current\n    return 2 * previous[-1] / (len(ref) + len(hyp))\n\n\ndef lexical_scores(reference, hypothesis):\n    """One example\'s EM, SQuAD-style Token-F1, BLEU-1/2 and ROUGE-L F1."""\n    ref, hyp = _as_text(reference), _as_text(hypothesis)\n    return {\n        "em": float(ref.casefold() == hyp.casefold()),\n        "token_f1": _token_f1(ref, hyp),\n        "bleu_1": _bleu(ref, hyp, 1),\n        "bleu_2": _bleu(ref, hyp, 2),\n        "rouge_l": _rouge_l(ref, hyp),\n    }\n\n\ndef build_bertscore_scorer(model_type="bert-base-uncased", device="cpu", batch_size=16,\n                           rescale_with_baseline=True):\n    """Create a reusable frozen semantic scorer with explicit settings."""\n    try:\n        from bert_score import BERTScorer\n    except ImportError as exc:\n        raise RuntimeError("BERTScore requires `pip install bert-score` or `pip install -r requirements.txt`") from exc\n    return BERTScorer(\n        model_type=model_type, lang="en", device=device, batch_size=batch_size,\n        rescale_with_baseline=rescale_with_baseline,\n    )\n\n\ndef score_pairs(references, hypotheses, bert_scorer):\n    """Score aligned generated answers once; return per-example metric rows."""\n    if len(references) != len(hypotheses):\n        raise ValueError("References and hypotheses must have equal lengths")\n    if bert_scorer is None:\n        raise ValueError("A BERTScore scorer is required for the six paper metrics")\n    rows = [lexical_scores(ref, hyp) for ref, hyp in zip(references, hypotheses)]\n    nonempty = [i for i, (ref, hyp) in enumerate(zip(references, hypotheses))\n                if _as_text(ref).strip() and _as_text(hyp).strip()]\n    nonempty_indices = set(nonempty)\n    for i, (ref, hyp) in enumerate(zip(references, hypotheses)):\n        if i not in nonempty_indices:\n            rows[i]["bertscore_f1"] = float(not _as_text(ref).strip() and\n                                           not _as_text(hyp).strip())\n    if nonempty:\n        _, _, f1 = bert_scorer.score([_as_text(hypotheses[i]) for i in nonempty],\n                                     [_as_text(references[i]) for i in nonempty])\n        values = f1.detach().cpu().tolist() if hasattr(f1, "detach") else list(f1)\n        if len(values) != len(nonempty):\n            raise RuntimeError("BERTScore returned the wrong number of examples")\n        for i, value in zip(nonempty, values):\n            rows[i]["bertscore_f1"] = float(value)\n    return rows\n\n\ndef mean_scores(rows, metrics=PAPER_METRICS):\n    if not rows:\n        raise ValueError("Cannot aggregate an empty evaluation set")\n    return {metric: fmean(row[metric] for row in rows) for metric in metrics}\n'}

EVAL_ROOT.mkdir(parents=True, exist_ok=True)
for name, source in BUNDLED_SOURCES.items():
    destination = EVAL_ROOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
for package in ("scripts", "utils"):
    (EVAL_ROOT / package / "__init__.py").write_text("", encoding="utf-8")
if str(EVAL_ROOT) not in sys.path:
    sys.path.insert(0, str(EVAL_ROOT))
from scripts.evaluate_kaggle_ministral import model_identifier
print("Evaluation scripts:", EVAL_ROOT)


## Build llama.cpp with CUDA

Compilation can take several minutes and is reused when rerunning cells
in the same session. The build targets your GPUs' compute capability.
Build errors stop this cell; evaluation never silently switches to CPU.
CUDA virtual memory management is disabled to avoid Kaggle's missing
`CUDA::cuda_driver` CMake target. The CUDA backend and GPU splitting remain enabled.


In [ ]:
LLAMA_ROOT = WORK / "llama.cpp"
if not LLAMA_ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", LLAMA_CPP_REF,
        "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_ROOT),
    ], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(LLAMA_ROOT), "rev-parse", "HEAD"], text=True).strip()
requested_commit = subprocess.check_output(
    ["git", "-C", str(LLAMA_ROOT), "rev-parse", f"{LLAMA_CPP_REF}^{{commit}}"], text=True).strip()
assert actual_commit == requested_commit, "Use a fresh working session after changing LLAMA_CPP_REF."
subprocess.run([
    "cmake", "-S", str(LLAMA_ROOT), "-B", str(LLAMA_ROOT / "build"),
    "-DGGML_CUDA=ON", "-DGGML_CUDA_NO_VMM=ON", f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCHS}",
    "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_OPENSSL=OFF",
], check=True)
subprocess.run([
    "cmake", "--build", str(LLAMA_ROOT / "build"), "--target", "llama-server",
    "--parallel", str(min(os.cpu_count() or 2, 4)),
], check=True)
LLAMA_SERVER = LLAMA_ROOT / "build/bin/llama-server"
subprocess.run([str(LLAMA_SERVER), "--list-devices"], check=True)
print("Built llama.cpp commit:", actual_commit)


## Run five questions first

The runner resolves local GGUF files or downloads them, starts a vision server, and
shuts it down when evaluation finishes or fails. Check predictions and
errors before relying on the full results. Smoke and full outputs use
separate directories; rerunning a smoke test resumes saved successes.
Each run selects a free local server port automatically, so the full run
can start immediately after smoke. Resume accepts port changes while
retaining the checks on model and evaluation settings.
The first cell below defines command arguments. Run it even when skipping
the smoke test during resume.


In [ ]:
def evaluation_command():
    command = [
        sys.executable, "-u", "-m", "scripts.evaluate_kaggle_ministral",
        "--llama-server", str(LLAMA_SERVER), "--model-dir", str(MODEL_DIR),
        "--cache-dir", str(WORK / "gguf-models"), "--hf-revision", HF_REVISION,
        "--context-size", str(CONTEXT_SIZE), "--dataset", DATASET,
        "--questions", str(QUESTIONS), "--images", str(IMAGES),
        "--image-split", IMAGE_SPLIT, "--max-tokens", str(MAX_TOKENS),
        "--bertscore-device", "cpu", "--timeout", "300",
    ]
    if NO_BERTSCORE:
        command.append("--no-bertscore")
    return command

def evaluation_output():
    return RESULTS_ROOT / f"{model_identifier(str(MODEL_DIR))}_{DATASET}"


In [ ]:
smoke_output = evaluation_output() / "smoke"
smoke = evaluation_command() + ["--limit", "5", "--output-dir", str(smoke_output),
                  "--server-log", str(WORK / "llama-smoke.log")]
if (smoke_output / "run_config.json").is_file():
    smoke.append("--resume")
subprocess.run(smoke, cwd=EVAL_ROOT, check=True)
smoke_report = json.loads((smoke_output / "report.json").read_text())
assert smoke_report["complete"] and smoke_report["errors"] == 0
print((smoke_output / "predictions.csv").read_text())


## Evaluate the full split

Each request uses only the image, question, and dataset instruction:
zero-shot, temperature 0, seed 42. GQA reports exact-match accuracy and
normalized accuracy; PlantExpert reports EM, Token-F1, BLEU-1/2,
ROUGE-L and BERTScore-F1. Predictions flush to JSONL after every request.

To resume, set `RESUME = True` and rerun the configuration, argument-setup
and full-run cells with the same dataset, paths, quantization, revisions
and settings. In a fresh session, run the dependency, bundled-script and
llama.cpp build cells too.
After a new Kaggle session, attach saved outputs and **copy the full run
directory back to the original `/kaggle/working` location** before resuming.
The smoke cell may be skipped when resuming. `run_config.json` records the
resolved HF commit, llama.cpp version and GPU split and rejects changed settings.


In [ ]:
# Rebuild the arguments so configuration edits also apply when resuming.
full_output = evaluation_output() / "full"
full = evaluation_command() + ["--output-dir", str(full_output),
                 "--server-log", str(WORK / "llama-full.log")]
if FULL_LIMIT is not None:
    full += ["--limit", str(FULL_LIMIT)]
if RESUME:
    full.append("--resume")
subprocess.run(full, cwd=EVAL_ROOT, check=True)


## Inspect and download results

Save a Kaggle notebook version to preserve output files. The archive
contains both runs' reports, CSV predictions, raw JSONL responses and
settings. GQA additionally exports `gqa_predictions.json`.

If inference fails, inspect `/kaggle/working/llama-full.log`. Responses
ending at the token limit are counted in `truncated_responses`; a high
count means a fresh run with a larger `MAX_TOKENS` may be needed.


In [ ]:
import shutil
from IPython.display import FileLink, display

report = json.loads((full_output / "report.json").read_text())
print(json.dumps(report, indent=2))
subprocess.run(["nvidia-smi"], check=True)  # Server has shut down; memory is released.
archive = shutil.make_archive(str(WORK / f"{model_identifier(str(MODEL_DIR))}_vqa_results"),
                              "zip", evaluation_output())
display(FileLink(archive))
